# Import question

In [ ]:
import pickle
question_data = "./data/dataset_all/new_all_questions_dict_wtq_nqt_wikisql.pkl"
with open(question_data,"rb") as f:
     new_question_corpus=pickle.load(f)


In [ ]:
import pickle
tables_data = "./data/dataset_all/all_table_transform_table_corpus.pkl"
with open(tables_data,"rb") as f:
     table_corpus=pickle.load(f)

In [ ]:
print(list(table_corpus["WIKISQL"].keys())[:5])

# Visualize

In [ ]:


import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from ..utils.core import *
from ..utils.model import encode_texts_in_batches


def _to_text(x):
    return x if isinstance(x, str) else str(x)

def _l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

@torch.no_grad()
def plot_one_table_all_from_origin(
    model_name,
    model,
    all_table_corpus,
    dataset="WTQ",
    table_id=None,
    dimension_reduction="tSNE",
    representations=None,
    origin_mode="centroid",     # "centroid" (recommended) or "none"
    normalize_cosine=True,      # recommended
    vis_scale=2.5,
    label_offset=0.06,
    batch_size=64,
    figsize=(8, 6),
    arrow_alpha=0.9,
    draw_label = False,
    point_size=45,
    label_fontsize=11,
    grid_alpha=0.25,
    random_state=0,
):
    """
    Plot all representations as vectors from a common origin.
    "Original" is treated like any other representation.

    origin_mode:
      - "centroid": subtract mean embedding across reps for this table (best)
      - "none": do not center (origin is arbitrary; less interpretable)
    """

    if representations is None:
        representations = [
            #Popular Representation
            'pipe_serialized','token_serialized','space_serialized',
            # Data Representation
            'csv','tsv','html','markdown','latex','dict','json','xml',
            # Structural Transformations 
            'shuffled_rows','shuffled_cols','transpose',
            #Schema and Definition Types
            'mschema','macschema','ddl',
        ]
    category = {
            "Popular Representation": ("centroid_popular",["pipe_serialized", "token_serialized", "space_serialized"],[0,1,2], "red"),
            "Data Representation": ("centroid_data",["csv", "tsv", "html", "markdown", "latex", "dict", "json", "xml"],[3,4,5,6,7,8,9,10],"blue"),
            "Structural Transformations": ("centroid_structural",["shuffled_rows", "shuffled_cols", "transpose"],[11,12,13],"green"),
            "Schema and Definition Types": ("centroid_schema",["mschema", "macschema", "ddl"],[14,15,16],"purple"),
    }
    table_corpus = all_table_corpus[dataset]
    if table_id is None:
        table_id = next(iter(table_corpus.keys()))

    # ---- Collect texts
    rep_texts, rep_names, missing = [], [], []
    for rep in representations:
        if rep not in table_corpus[table_id]:
            missing.append(rep)
            continue
        rep_texts.append(_to_text(table_corpus[table_id][rep]))
        rep_names.append(rep)

    if len(rep_texts) < 2:
        raise ValueError(f"Need >=2 reps. Got {len(rep_texts)}. Missing={missing}")

    # ---- Embed
    embs = encode_texts_in_batches(
        model_name,
        model,
        rep_texts,
        instruction="",
        batch_size=batch_size,
    )
    
    if isinstance(embs, np.ndarray):
        embs = torch.from_numpy(embs)
    if model_name == "splade":
        embs = embs.to_dense()
    embs = embs.float().cpu().numpy()  # (R, D)

    # ---- Center to make a meaningful common origin
    X = embs.copy()

    # ===================== NEW: colors + category centroids (+ centroid_all) =====================
    # Build per-representation colors (default gray if not in any category)
    rep_colors = ["gray"] * len(rep_names)
    for _, (_, _, idxs, color) in category.items():
        for j in idxs:
            if 0 <= j < len(rep_names):
                rep_colors[j] = color

    # Compute category centroids from X using provided indices (only valid indices)
    centroid_names = []
    centroid_vecs = []
    centroid_colors = []

    for _, (cent_name, _, idxs, color) in category.items():
        valid = [j for j in idxs if 0 <= j < X.shape[0]]
        if len(valid) == 0:
            continue
        centroid_vecs.append(X[valid].mean(axis=0))
        centroid_names.append(cent_name)
        centroid_colors.append(color)

    # centroid_all over all available reps
    centroid_all = X.mean(axis=0)
    centroid_vecs.append(centroid_all)
    centroid_names.append("centroid_all")
    centroid_colors.append("black")  # choose a neutral color for "all"

    # Augment X / names / colors with centroid vectors
    if len(centroid_vecs) > 0:
        X = np.vstack([X, np.stack(centroid_vecs, axis=0)])
        rep_names = rep_names + centroid_names
        rep_colors = rep_colors + centroid_colors


    if origin_mode.lower() == "centroid":
        mu = X.mean(axis=0, keepdims=True)
        X = X - mu
    elif origin_mode.lower() == "none":
        pass
    else:
        raise ValueError("origin_mode must be 'centroid' or 'none'")

    # ---- Cosine normalization (direction-focused)
    if normalize_cosine:
        X = _l2_normalize_rows(X)

    if dimension_reduction == "PCA":
        # ---- PCA to 2D
        Z = PCA(n_components=2, random_state=random_state).fit_transform(X)  # (R, 2)
    elif dimension_reduction == "tSNE":
        n = X.shape[0]  # 17
        perp = min(5, n - 1)
        X_50 = PCA(n_components=min(50, X.shape[0]-1), random_state=random_state).fit_transform(X)

        Z = TSNE(
            n_components=2,
            perplexity=perp,
            init="pca",
            learning_rate="auto",
            random_state=random_state,
        ).fit_transform(X_50)


    # ---- Visual scaling
    Z_vis = Z * float(vis_scale)

    # ---- Plot
    fig, ax = plt.subplots(figsize=figsize)
    

    # Arrow head sizing
    span = max(Z_vis[:,0].max() - Z_vis[:,0].min(), Z_vis[:,1].max() - Z_vis[:,1].min())
    head_w = 0.02 * (span + 1e-9)
    head_l = 0.03 * (span + 1e-9)

    # Draw origin point
    ax.scatter([0], [0], s=point_size)
    ax.text(0, 0, " origin", fontsize=label_fontsize, va="center")

    # Draw arrows from (0,0) to each rep
    for i, name in enumerate(rep_names):
        xi, yi = Z_vis[i]
        if "centroid" in name:
            ax.scatter(xi, yi, s=point_size+5,color="black",marker="x")
        else:
            ax.scatter(xi, yi, s=point_size,color=rep_colors[i])
        ax.arrow(
            0.0, 0.0, xi, yi,
            length_includes_head=True,
            head_width=head_w,
            head_length=head_l,
            alpha=arrow_alpha,
            linewidth=1.6,
            color="black"  if "centroid" in name else rep_colors[i] ,
            hatch="o"  if "centroid" in name else "x" 
        )
        if draw_label:
            # label slightly offset along its direction
            v = np.array([xi, yi], dtype=float)
            n = np.linalg.norm(v) + 1e-9
            ux, uy = v / n
            ax.text(
                xi + label_offset * ux,
                yi + label_offset * uy,
                name,
                fontsize=label_fontsize,
                va="center",
                ha="left"
            )
            print( xi + label_offset * ux,
                yi + label_offset * uy,
                name,)

    '''ax.set_title(
        f"Dataset = {dataset} | Table Id = {table_id}"
       # f"origin={origin_mode} | cosine_norm={normalize_cosine} | scale={vis_scale}"
    )'''
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    ax.axhline(0, linewidth=0.6)
    ax.axvline(0, linewidth=0.6)
    ax.grid(True, linewidth=0.5, alpha=grid_alpha)

    # Padding so labels don’t clip
    pad = 0.12 * (span + 1e-9)
    ax.set_xlim(Z_vis[:,0].min() - pad, Z_vis[:,0].max() + pad)
    ax.set_ylim(Z_vis[:,1].min() - pad, Z_vis[:,1].max() + pad)

    #plt.tight_layout()

    if missing:
        print(f"[plot] Missing reps for table_id={table_id}: {missing}")

    return fig, ax, {"table_id": table_id, "rep_names": rep_names, "Z": Z, "Z_vis": Z_vis}




In [ ]:
from transformers import AutoModel
from accelerate import Accelerator
import torch
from ..utils.model import get_model

import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"
model_name="reasonir"

accelerator = Accelerator()

# get ReasonIR model
model = get_model(accelerator,model_name=model_name)

In [ ]:
fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id='csv/200-csv/0',
    dimension_reduction="PCA",
    origin_mode="none",     # <-- key change
    normalize_cosine=True,
    draw_label=True,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

'''fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id='csv/200-csv/1',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id= 'csv/200-csv/10',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WTQ",
    table_id= 'csv/200-csv/11',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()'''

In [ ]:
fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id='1-10015132-16',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

'''fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id= '1-10083598-1',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id= '1-1013129-2',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="WIKISQL",
    table_id= '1-1013129-3',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()'''

In [ ]:
fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id='Lesley Joseph_A1D55A57012E3362',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

'''fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id='List of India national cricket captains_DF24C9CC57542D09',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()
fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id= 'Infinity on High_C66E938E06D39ECB',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()

fig, ax, info = plot_one_table_all_from_origin(
    model_name=model_name,
    model=model,
    all_table_corpus=table_corpus,
    dataset="NQ",
    table_id= 'Hydrogen sulfide_54EB24287C466E6E',
    dimension_reduction="tSNE",
    origin_mode="none",     # <-- key change
    normalize_cosine=False,
    vis_scale=2.8,
    label_offset=0.07,
)
plt.show()'''